# Prepare dataset

In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
np.random.seed(1337)
random.seed(1337)

In [3]:
df = pd.read_csv("./student_habits_performance.csv")
df = pd.DataFrame(df)
df.head()

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [4]:
df['parental_education_level'] = df['parental_education_level'].fillna(
    df['parental_education_level'].mode()[0]
)


In [5]:
cat_col = df.select_dtypes(include='object').columns
cat_col = [col for col in cat_col]
cat_col.remove('student_id')
cat_col

['gender',
 'part_time_job',
 'diet_quality',
 'parental_education_level',
 'internet_quality',
 'extracurricular_participation']

In [6]:
num_col = df.select_dtypes(exclude='object').columns
num_col = [col for col in num_col]
num_col

['age',
 'study_hours_per_day',
 'social_media_hours',
 'netflix_hours',
 'attendance_percentage',
 'sleep_hours',
 'exercise_frequency',
 'mental_health_rating',
 'exam_score']

In [7]:
df2 = df.drop('student_id', axis=1)
df2.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [8]:
diet_quality = {'Poor': 0, 'Fair': 1, 'Good': 2}
parental_education_level = {'High School': 0, 'Bachelor': 1, 'Master': 2}
internet_quality = {'Poor': 0, 'Average': 1, 'Good': 2}

df2['dq_e'] = df2['diet_quality'].map(diet_quality)
df2['pel_e'] = df2['parental_education_level'].map(parental_education_level)
df2['iq_e'] = df2['internet_quality'].map(internet_quality)


In [9]:
dummies = pd.get_dummies(
    df[['gender', 'part_time_job', 'extracurricular_participation']],
    drop_first=True,
)


In [10]:
df3 = pd.concat([df2, dummies], axis=1)
df3 = df3.drop(
    [
        'gender',
        'part_time_job',
        'diet_quality',
        'parental_education_level',
        'internet_quality',
        'extracurricular_participation',
    ],
    axis=1,
)


In [11]:
df4 = df3

In [12]:
X = df4.drop('exam_score', axis=1)
y = df4['exam_score']
X.shape, y.shape

((1000, 15), (1000,))

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape  # this will show the shape after scaling

(1000, 15)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

X_train = X_train.tolist()
X_test = X_test.tolist()
y_train = y_train.tolist()
y_test = y_test.tolist()


# Training demo

In [ ]:
from micrograd import nn


def train(
    model: nn.MLP,
    loss_fn: nn.MSELoss,
    x_train: list,
    y_train: list,
    # lr: float = 5e-3,
    num_steps: int = 200,
):
    for step in range(num_steps):
        y_pred = [model(x) for x in x_train]

        loss = loss_fn(y_pred, y_train)

        if step % 10 == 0:
            print(f'Step: {step}. Loss: {loss.data}')

        model.zero_grad()
        loss.backward()

        lr = 0.04 - 0.049 * step / num_steps
        for p in model.parameters():
            p.data = p.data - lr * p.grad


model = nn.MLP(15, [1])
loss_fn = nn.MSELoss()


print("Num parameter", len(model.parameters()))

train(model, loss_fn, X_train, y_train)

Num parameter 16
Step: 0. Loss: 5075.938745326474
Step: 10. Loss: 1014.5772321836848
Step: 20. Loss: 244.17265048989714
Step: 30. Loss: 81.31242947166754
Step: 40. Loss: 43.05706288263705
Step: 50. Loss: 33.09804190269677
Step: 60. Loss: 30.231953679511104
Step: 70. Loss: 29.32252787433599
Step: 80. Loss: 29.005296684033244
Step: 90. Loss: 28.88406877204227
Step: 100. Loss: 28.833540906299014
Step: 110. Loss: 28.810708203453657
Step: 120. Loss: 28.799620290029917
Step: 130. Loss: 28.793916245148903
Step: 140. Loss: 28.790889318950732
Step: 150. Loss: 28.789329788884704
Step: 160. Loss: 28.788694767624616
Step: 170. Loss: 28.788780611557982
Step: 180. Loss: 28.789609797446847
Step: 190. Loss: 28.791440738428374


In [17]:
from sklearn.metrics import r2_score

y_pred = [model(x) for x in X_test]
y_pred = [y_.data for y_ in y_pred]
print(
    f'R²: {r2_score(y_test, y_pred):.3f}'
)

R²: 0.899
